# MangoLeaf AI V4.3.0: fit one combined seven-class rejection gate

**Experimental candidate workflow. Run results are pending.** Run this after the training section of `MangoLeaf_AI_Training_v4_3_0.ipynb` saves `training_complete.json`. That notebook trains one combined model with seven classes and includes all figures. This notebook fixes the neural-network weights and fits the decision rules used after prediction. **Gate fitting is not neural-network training.** It never calls `model.fit` or changes saved training files.

The confidence floor stays **0.70**. Only the margin and normalized entropy gates are searched, using **mango validation + the original old OOD fitting subset**. The gate must retain at least **95% of validation mango images**, including any accepted wrong predictions. Selection is hashed and frozen before any mango test or OOD report inference. An infeasible selection remains a failure; confidence-only reporting cannot become a fitted candidate.

`mango_ood_test` and `mango_ood_test_new_photos` are the original reused OOD folders, distinct from the reviewed training background folder. The historically named “new” folder contained 32 photos already evaluated in V4.1/V4.1.1. It is report-only in this run and is **not pristine external evidence**. Audited exclusions may reduce its retained count. Old mango test and old OOD reports are also exploratory reuse. Do not tune from these reports. Field diagnosis accuracy and browser parity need separate evaluation.

Run Step 1–12 in order. After Step 12, return to the matching training notebook's figure section with the printed threshold run path. Keep both complete Drive run folders for provenance. Every code cell is clean and unexecuted in this template.

## Step 1: Set paths and runtime

**Purpose:** Choose the completed combined training run and the historical OOD reference before reading images.

**Reads:** Google Drive; the completed V4.3.0 training run; original V4.1.1 evaluation_inputs.json.

**Saves:** Runtime settings in memory. No experiment output folder yet.

**Check:** TRAINING_RUN accepts the completed run folder name or its full absolute Drive path under V4.3.0/training. Keep both OOD paths separate from training backgrounds. List exact content-based exclusions before the audit.

**Expected result:** Seven class names, one combined arm, confidence 0.70, minimum retention 0.95, and paths print below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import csv, gc, hashlib, json, math, random, shutil, sys, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps, ImageDraw
from scipy.fftpack import dct

SEED = 42
EXPERIMENT_VERSION = 'v4.3.0'
MODEL_VERSION = 'model-v4.3.0-candidate'
ARMS = ('combined',)
CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold']
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.70
MIN_ID_RETENTION = 0.95  # Acceptance includes both correct and wrong mango predictions.
GRID = [i / 100 for i in range(101)]
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
PREPROCESSING = 'TensorFlow RGB decode (Pillow RGB with EXIF transpose for WebP); bilinear 224x224, no antialias; float32 0-255; model rescales internally'
CONFIDENCE_ONLY = {'confidenceThreshold': 0.70, 'marginThreshold': 0.0, 'entropyThreshold': 1.0}

EXPERIMENT_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.3.0')
TRAINING_RUN = 'PASTE_v4-3-0-training_RUN_FOLDER'  # Completed folder name, or full absolute Drive path.
OLD_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test')
NEW_OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test_new_photos')
REFERENCE_V411_INPUTS = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.1.1/threshold_fitting/v4-1-1-thresholds-20260928T060638Z-a3a7418c/evaluation_inputs.json')
REFERENCE_V411_SHA256 = '9692d77f942abd2a2715b0f9f6bfc7a19833be421711c065b9bbe7ab0fe129a9'
OUTPUT_ROOT = EXPERIMENT_ROOT / 'threshold_fitting'
OLD_OOD_EXCLUDE = []  # Exact relative paths; keep historical tune/report assignment first.
NEW_OOD_EXCLUDE = []  # Exact relative paths, for example an image actually containing mango.
EXCLUSION_REASONS = {}  # {'old/path.jpg': 'Content reason', 'new/path.jpg': 'Content reason'}


def resolve_training_run(value):
    text = str(value).strip()
    if not text:
        raise ValueError('Set TRAINING_RUN to a completed run folder name or its full absolute Drive path.')
    path = Path(text)
    root = (EXPERIMENT_ROOT / 'training').resolve()
    if not path.is_absolute():
        if len(path.parts) != 1 or path.name in ('.', '..'):
            raise ValueError('Nested relative TRAINING_RUN paths are ambiguous. Use the folder name or full absolute Drive path.')
        path = root / path
    path = path.resolve()
    if path.parent != root:
        raise ValueError('TRAINING_RUN must be one completed folder directly under ' + str(root))
    if not path.is_dir() or not (path / 'training_complete.json').is_file():
        raise FileNotFoundError('Completed V4.3.0 training run not found: ' + str(path)
                                + '. Paste the folder printed by the training notebook.')
    return path


TRAINING_RUN = resolve_training_run(TRAINING_RUN)
input_roots = [TRAINING_RUN.resolve(), OLD_OOD_DIR.resolve(), NEW_OOD_DIR.resolve()]
if len(set(input_roots)) != len(input_roots):
    raise ValueError('Training and both OOD input folders must be distinct.')
for input_root in input_roots:
    if (input_root.is_relative_to(OUTPUT_ROOT.resolve())
            or OUTPUT_ROOT.resolve().is_relative_to(input_root)):
        raise ValueError('Threshold output and input folders cannot contain one another.')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.keras.mixed_precision.set_global_policy('float32')
RUNTIME = {'python': sys.version, 'tensorflow': tf.__version__,
           'keras': tf.keras.__version__, 'numpy': np.__version__, 'pillow': Image.__version__}
print('Runtime:', RUNTIME)
print('Model and experiment:', MODEL_VERSION, EXPERIMENT_VERSION)
print('Classes:', CLASS_NAMES, '| arms:', ARMS)
print('Confidence:', CONFIDENCE_THRESHOLD, '| minimum validation retention:', MIN_ID_RETENTION)
print('Training run:', TRAINING_RUN)
print('Threshold output root:', OUTPUT_ROOT)


## Step 2: Define file and split checks

**Purpose:** Stop on changed provenance, unsafe paths, or mismatched split files.

**Reads:** SHA-256 digests, training artifact manifest, and recorded dataset ZIP members when the helpers are called.

**Saves:** Helper definitions only.

**Check:** The ZIP extractor accepts only manifest-listed images inside its new temporary folder.

**Expected result:** The functions are defined without extracting or predicting yet.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def object_hash(value):
    text = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(text.encode()).hexdigest()


def write_json_once(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write('\n')


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder missing: {folder}')
    paths = sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise ValueError(f'No supported images in {folder}')
    return paths


def verify_records_unchanged(rows):
    for row in rows:
        if sha256_file(row['path']) != row['sha256']:
            raise RuntimeError('Image changed: ' + row['path'])


def inside(root, relative):
    relative = PurePosixPath(relative)
    if relative.is_absolute() or '..' in relative.parts or '\\' in str(relative):
        raise ValueError('Unsafe relative path: ' + str(relative))
    target = (Path(root) / str(relative)).resolve()
    if not target.is_relative_to(Path(root).resolve()) or target == Path(root).resolve():
        raise ValueError('Path must stay inside its output folder.')
    return target


def extract_dataset(archive, rows, destination):
    # Extract only recorded images, then verify their contents before inference.
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=False)
    expected = {r['relativePath']: r for r in rows}
    if len(expected) != len(rows):
        raise ValueError('The image manifest repeats a destination path.')
    with zipfile.ZipFile(archive) as bundle:
        names = bundle.namelist()
        if len(names) != len(set(names)) or set(names) != set(expected):
            raise ValueError('Dataset ZIP entries differ from the image manifest.')
        for name, row in expected.items():
            target = inside(destination, name)
            if bundle.getinfo(name).file_size != row['bytes']:
                raise ValueError('Image size changed in dataset ZIP: ' + name)
            target.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(name) as source, target.open('xb') as output:
                shutil.copyfileobj(source, output)
            if sha256_file(target) != row['sha256']:
                raise ValueError('Image contents changed in dataset ZIP: ' + name)
    return [{**r, 'path': str(inside(destination, r['relativePath']))} for r in rows]


def verify_artifacts(folder, artifacts):
    for relative, expected in artifacts.items():
        path = inside(folder, relative)
        if not path.is_file() or sha256_file(path) != expected:
            raise RuntimeError('Missing or changed saved artifact: ' + relative)


## Step 3: Define image and model prediction helpers

**Purpose:** Use exactly the saved seven-class training input contract.

**Reads:** RGB images, calibrated Keras weights, and original hashes when called.

**Saves:** Probability arrays in memory; helper definitions alone save nothing.

**Check:** Pixels remain float32 0–255; the model rescales internally. Temperature was already folded into its weights.

**Expected result:** Validation predictions can later be compared with the saved training predictions.

In [ ]:
def load_pixels(path):
    # JPEG/PNG/GIF/BMP use TensorFlow decode; WebP uses Pillow RGB. Keep 0-255 pixels.
    if Path(path).suffix.lower() == '.webp':
        # TensorFlow decode_image cannot decode WebP; match the training notebook's RGB branch.
        with Image.open(path) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert('RGB'), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method='bilinear',
                           preserve_aspect_ratio=False, antialias=False)


def predict_records(model, rows):
    output = []
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        verify_records_unchanged(batch)
        pixels = tf.stack([load_pixels(r['path']) for r in batch])
        values = model(pixels, training=False).numpy()
        if values.shape != (len(batch), len(CLASS_NAMES)) or not np.isfinite(values).all():
            raise ValueError('The model must return seven finite probabilities per image.')
        output.extend(values.tolist())
    if not output:
        raise ValueError('Cannot predict an empty image collection.')
    return np.asarray(output, dtype=np.float32)


def weight_digest(model):
    digest = hashlib.sha256()
    for variable in model.weights:
        values = variable.numpy()
        digest.update(str(values.shape).encode())
        digest.update(str(values.dtype).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()


def print_class_probabilities(probabilities, title):
    # Sort label-score pairs for display; keep the model's class order unchanged.
    ranked = sorted(zip(CLASS_NAMES, map(float, probabilities), strict=True),
                    key=lambda item: item[1], reverse=True)
    print('\n' + title)
    for label, probability in ranked:
        print(f"{label.replace('_', ' ').title():<24} {probability * 100:6.2f}%")
def verify_all_records_unchanged(rows):
    verify_records_unchanged(rows)
    for row in rows:
        if Path(row['path']).stat().st_size != row['bytes']:
            raise RuntimeError('Image size changed after audit: ' + row['path'])


def load_arm(arm):
    info = arm_infos[arm]
    if sha256_file(info['modelPath']) != info['modelSha256']:
        raise RuntimeError('Model changed after audit: ' + arm)
    if sha256_file(info['infoPath']) != info['infoSha256']:
        raise RuntimeError('Calibration metadata changed after audit: ' + arm)
    model = tf.keras.models.load_model(info['modelPath'], compile=False, safe_mode=True)
    if tuple(model.input_shape[1:]) != (224, 224, 3) or tuple(model.output_shape[1:]) != (7,):
        raise ValueError('Unexpected model input/output shape: ' + arm)
    return model


def verify_frozen_inputs():
    if sha256_file(TRAINING_RUN / 'training_complete.json') != COMPLETE_SHA256:
        raise RuntimeError('Training completion record changed.')
    if sha256_file(TRAINING_RUN / 'input_manifest.json') != TRAIN_MANIFEST_SHA256:
        raise RuntimeError('Training input manifest changed.')
    if sha256_file(TRAINING_RUN / 'dataset_split.zip') != complete['datasetSplitSha256']:
        raise RuntimeError('Saved split changed.')
    if sha256_file(INPUT_PATH) != INPUT_SHA256:
        raise RuntimeError('Evaluation input manifest changed.')
    if sha256_file(REFERENCE_V411_INPUTS) != REFERENCE_V411_INPUTS_SHA256:
        raise RuntimeError('Historical OOD reference changed after audit.')
    if sha256_file(TRAINING_RUN / 'plan.json') != complete['planSha256']:
        raise RuntimeError('Training plan changed after audit.')
    verify_artifacts(TRAINING_RUN, complete['artifacts'])
    verify_all_records_unchanged(all_records)
    for arm in ARMS:
        info = arm_infos[arm]
        if sha256_file(info['modelPath']) != info['modelSha256']:
            raise RuntimeError('Model weights changed: ' + arm)
        if sha256_file(info['infoPath']) != info['infoSha256']:
            raise RuntimeError('Model metadata changed: ' + arm)


## Step 4: Define browser model file checks

**Purpose:** Check the complete TensorFlow.js graph and weight manifest before packaging.

**Reads:** Training model.json and its listed unquantized weight shards.

**Saves:** Helper definitions and a verified file list when called.

**Check:** One 224 × 224 RGB input and exactly seven output scores; each shard path stays inside the export folder.

**Expected result:** Broken signatures, unsafe/repeated paths, truncated weights, and nonfinite weights stop packaging.

In [ ]:
def verify_tfjs_export(folder):
    folder = Path(folder)
    graph = json.loads((folder / 'model.json').read_text(encoding='utf-8'))
    if graph.get('format') != 'graph-model':
        raise ValueError('Expected a TensorFlow.js Graph Model.')
    signature = graph.get('signature', {})
    outputs = list(signature.get('outputs', {}).values())
    inputs = list(signature.get('inputs', {}).values())
    dims = lambda tensor: [int(d['size']) for d in tensor['tensorShape']['dim']]
    if len(outputs) != 1 or dims(outputs[0])[-1:] != [len(CLASS_NAMES)]:
        raise ValueError('Export must expose seven output probabilities.')
    if len(inputs) != 1 or dims(inputs[0])[1:] != [224, 224, 3]:
        raise ValueError('Export input shape differs from the training plan.')
    dtypes = {'float32': '<f4', 'int32': '<i4', 'bool': '?'}
    used = set()
    for group in graph['weightsManifest']:
        parts = []
        for name in group['paths']:
            if name in used:
                raise ValueError('Repeated weight shard: ' + name)
            used.add(name)
            parts.append(inside(folder, name).read_bytes())
        buffer, offset = b''.join(parts), 0
        for weight in group['weights']:
            if 'quantization' in weight or weight['dtype'] not in dtypes:
                raise ValueError('This experiment requires unquantized float32 weights.')
            dtype = np.dtype(dtypes[weight['dtype']])
            count = math.prod(weight['shape'])
            size = count * dtype.itemsize
            if offset + size > len(buffer):
                raise ValueError('Truncated weight shard.')
            values = np.frombuffer(buffer[offset:offset + size], dtype=dtype)
            if not np.isfinite(values).all():
                raise ValueError('Non-finite values found in the exported weights.')
            offset += size
        if offset != len(buffer):
            raise ValueError('Weight manifest and shard sizes disagree.')
    if not used:
        raise ValueError('No exported weight shards.')
    return ['model.json'] + sorted(used)


## Step 5: Define rejection gates and report metrics

**Purpose:** Separate raw prediction accuracy from acceptance, rejection, and OOD false accepts.

**Reads:** Seven-class probabilities; only validation and old OOD fitting scores enter select_thresholds.

**Saves:** Helper definitions; later a full 101 × 101 search table and full-precision per-image reports.

**Check:** Accepted means confidence ≥ 0.70 AND margin ≥ its gate AND entropy ≤ its gate. Ties favor greater retention, lower margin, then higher entropy.

**Expected result:** A gate is selected only if at least ceil(0.95 × validation count) images pass. Otherwise selection is null.

In [ ]:
def probability_stats(probabilities):
    stats = []
    for row in probabilities:
        values = [float(p) for p in row]
        if len(values) != len(CLASS_NAMES) or any(
                not math.isfinite(p) or p < 0 or p > 1 for p in values):
            raise ValueError('Expected seven finite probabilities in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Expected softmax probabilities summing to one.')
        ordered = sorted(values, reverse=True)
        stats.append({'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
                      'entropy': -sum(p * math.log(p) for p in values if p > 0)
                                 / math.log(len(values)),
                      'predictedIndex': values.index(ordered[0]),
                      # Keep full precision in CSVs; only preview text is rounded.
                      **{f'probability_{label}': p for label, p in zip(CLASS_NAMES, values)}})
    return stats


def accepted(stat, thresholds):
    return (stat['confidence'] >= thresholds['confidenceThreshold']
            and stat['margin'] >= thresholds['marginThreshold']
            and stat['entropy'] <= thresholds['entropyThreshold'])


def select_thresholds(val_stats, ood_tune_stats, grid, confidence, min_retention):
    # Only tuning data enters this function. Counts avoid rounded-score ties.
    if not val_stats or not ood_tune_stats:
        raise ValueError('Both tuning subsets must contain predictions.')
    if not 0 < min_retention <= 1 or not 0 <= confidence <= 1:
        raise ValueError('Invalid confidence or retention target.')
    if not grid or any(not math.isfinite(x) or not 0 <= x <= 1 for x in grid):
        raise ValueError('Grid must contain finite values between zero and one.')
    required = math.ceil(min_retention * len(val_stats) - 1e-9)
    best, best_key, rows = None, None, []
    for margin in grid:
        val_eligible = [s for s in val_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        ood_eligible = [s for s in ood_tune_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        for entropy in grid:
            kept = sum(s['entropy'] <= entropy for s in val_eligible)
            rejected = len(ood_tune_stats) - sum(s['entropy'] <= entropy for s in ood_eligible)
            feasible = kept >= required
            rows.append({'margin': margin, 'entropy': entropy, 'valAccepted': kept,
                         'valTotal': len(val_stats), 'oodRejected': rejected,
                         'oodTotal': len(ood_tune_stats), 'meetsRetentionTarget': feasible})
            # Ties: more mango retention, then lower margin, then higher entropy.
            key = (rejected, kept, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin,
                        'entropyThreshold': entropy}
    return best, rows
def scored_rows(records, stats, thresholds):
    if len(records) != len(stats):
        raise ValueError('Image and prediction counts differ.')
    rows = []
    for record, stat in zip(records, stats):
        keep = accepted(stat, thresholds)
        prediction = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']: reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']: reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']: reasons.append('entropy')
        rows.append({**record, **stat, 'predictedClass': prediction, 'accepted': keep,
                     'returnedClass': prediction if keep else 'uncertain',
                     'rejectionReasons': ','.join(reasons),
                     'acceptedCorrect': keep and prediction == record['label']})
    return rows


def summarize(rows):
    if not rows:
        raise ValueError('Cannot report an empty subset.')
    count = len(rows)
    kept = sum(row['accepted'] for row in rows)
    report = {'total': count, 'accepted': kept, 'rejected': count - kept,
              'retention': kept / count, 'rejectionRate': (count - kept) / count}
    if all(row['label'] == 'OOD' for row in rows):
        report['falseAcceptRate'] = kept / count
        return report
    if any(row['label'] == 'OOD' for row in rows):
        raise ValueError('Report mango and OOD subsets separately.')
    correct = sum(row['acceptedCorrect'] for row in rows)
    report.update({'rawAccuracy': sum(row['predictedClass'] == row['label'] for row in rows) / count,
                   'acceptedCorrect': correct, 'acceptedWrong': kept - correct,
                   'acceptedAccuracy': correct / kept if kept else None,
                   'acceptedErrorsPerInput': (kept - correct) / count})
    per_class = {}
    confusion = [[0] * (len(CLASS_NAMES) + 1) for _ in CLASS_NAMES]
    for row in rows:
        column = row['predictedIndex'] if row['accepted'] else len(CLASS_NAMES)
        confusion[CLASS_NAMES.index(row['label'])][column] += 1
    for label in CLASS_NAMES:
        group = [row for row in rows if row['label'] == label]
        class_kept = sum(row['accepted'] for row in group)
        class_correct = sum(row['acceptedCorrect'] for row in group)
        per_class[label] = {'total': len(group), 'acceptedCorrect': class_correct,
                            'acceptedWrong': class_kept - class_correct,
                            'rejected': len(group) - class_kept,
                            'retention': class_kept / len(group) if group else None}
    healthy = {'healthy_leaf', 'healthy_fruit'}
    report['healthyErrorCounts'] = {
        'healthyAcceptedAsDisease': sum(r['accepted'] and r['label'] in healthy and r['predictedClass'] not in healthy for r in rows),
        'diseaseAcceptedAsHealthy': sum(r['accepted'] and r['label'] not in healthy and r['predictedClass'] in healthy for r in rows),
        'healthyOrganConfusion': sum(r['accepted'] and r['label'] in healthy and r['predictedClass'] in healthy and r['label'] != r['predictedClass'] for r in rows)}
    report['perClass'] = per_class
    report['confusion'] = {'rows': CLASS_NAMES, 'columns': CLASS_NAMES + ['uncertain'], 'counts': confusion}
    return report
def write_csv_once(path, rows):
    if not rows:
        raise ValueError('Cannot write an empty report.')
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def read_frozen_selection(path, expected_sha256):
    if sha256_file(path) != expected_sha256:
        raise RuntimeError('Saved selection changed. Start a new run; do not retune during evaluation.')
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_selection_lock():
    # Persist the selected digest before held-out inference; do not trust a new in-memory hash.
    lock = json.loads((RUN_DIR / 'selection_lock.json').read_text(encoding='utf-8'))
    if (lock['selectionSha256'] != SELECTION_SHA256 or lock['evaluationInputsSha256'] != INPUT_SHA256
            or lock['trainingCompleteSha256'] != COMPLETE_SHA256
            or lock['lockedBeforeReportInference'] is not True):
        raise RuntimeError('Selection lock differs from this audited run.')
    return read_frozen_selection(SELECTION_PATH, lock['selectionSha256'])


## Step 6: Define OOD scope and overlap audits

**Purpose:** Preserve the historical old OOD split and make visual review possible before fitting.

**Reads:** Original OOD photos, reviewed exclusions, mango splits, and training background source records.

**Saves:** Helper definitions; later contact-sheet images and indexes, exact/near-duplicate audit records.

**Check:** Old OOD split assignment happens before exclusions. Exact overlaps stop the run; near pairs require individual written review.

**Expected result:** Every old/new OOD photo can be inspected, including excluded photos. No OOD inference runs in the audit.

In [ ]:
def reviewed_paths(root):
    """Reject symlink escapes and unsupported folder layouts before any predictions."""
    root = Path(root).resolve()
    unsupported = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()
                         and p.suffix.lower() in {'.heic', '.avif', '.tif', '.tiff'})
    if unsupported:
        raise ValueError('Convert unsupported image format to JPEG or PNG, then start a new run: '
                         + ', '.join(unsupported[:5]))
    paths = list_images(root)
    if any(not p.resolve().is_relative_to(root) for p in paths):
        raise ValueError('An image symlink leaves its source folder.')
    return paths


def image_record(root, path, subset):
    with Image.open(path) as picture:
        picture.verify()
    return {'subset': subset, 'label': 'OOD', 'path': str(path.resolve()),
            'relativePath': path.relative_to(root).as_posix(),
            'sha256': sha256_file(path), 'bytes': path.stat().st_size}


def prepare_old_ood(root, excluded):
    root = Path(root).resolve()
    paths = reviewed_paths(root)
    all_names = {p.relative_to(root).as_posix() for p in paths}
    if len(excluded) != len(set(excluded)) or not set(excluded) <= all_names:
        raise ValueError('OLD_OOD_EXCLUDE repeats or names a missing image.')
    tune_dir, report_dir = root / 'tune', root / 'report'
    if tune_dir.is_dir() != report_dir.is_dir():
        raise ValueError('Old OOD needs both tune/ and report/, or neither.')
    if tune_dir.is_dir():
        if any(p.relative_to(root).parts[0] not in ('tune', 'report') for p in paths):
            raise ValueError('Old OOD photos outside tune/report would be ignored.')
        assigned = [('ood_tune', p) if p.is_relative_to(tune_dir) else ('ood_report', p)
                    for p in paths]
        method = 'existing tune/ and report/ directories'
    else:
        if any(p.parent != root for p in paths):
            raise ValueError('Old OOD must be flat or have explicit tune/report folders.')
        # Match V4 historical assignment: sorted full path, RandomState(42), half for tune.
        paths = sorted(paths)
        order = np.random.RandomState(SEED).permutation(len(paths))
        half = len(paths) // 2
        tune = {int(i) for i in order[:half]}
        assigned = [('ood_tune' if i in tune else 'ood_report', p) for i, p in enumerate(paths)]
        method = 'historical V4 flat-folder split: sorted paths, RandomState(42), 50/50'
    rows = [image_record(root, p, subset) for subset, p in assigned
            if p.relative_to(root).as_posix() not in excluded]
    if not any(r['subset'] == 'ood_tune' for r in rows) or not any(r['subset'] == 'ood_report' for r in rows):
        raise ValueError('Old OOD tune and report both need retained images.')
    return rows, method


def prepare_new_ood(root, excluded):
    root = Path(root).resolve()
    paths = reviewed_paths(root)
    all_names = {p.relative_to(root).as_posix() for p in paths}
    if len(excluded) != len(set(excluded)) or not set(excluded) <= all_names:
        raise ValueError('NEW_OOD_EXCLUDE repeats or names a missing image.')
    rows = [image_record(root, p, 'ood_new_report') for p in paths
            if p.relative_to(root).as_posix() not in excluded]
    if not rows:
        raise ValueError('New OOD report needs at least one eligible image.')
    return rows, paths
def perceptual_hashes(path):
    """Compare all right-angle rotations and mirrors; pHash flags need human review."""
    with Image.open(path) as picture:
        base = ImageOps.exif_transpose(picture).convert('L')
    rotations = [base.rotate(angle, expand=True) for angle in (0, 90, 180, 270)]
    views = rotations + [ImageOps.mirror(view) for view in rotations]
    hashes = []
    for view in views:
        pixels = np.asarray(view.resize((32, 32)), dtype=np.float32)
        frequency = dct(dct(pixels, axis=0, norm='ortho'), axis=1, norm='ortho')[:8, :8]
        median = np.median(frequency.flatten()[1:])
        hashes.append(sum(int(value > median) << i for i, value in enumerate(frequency.flatten())))
    return hashes


def audit_duplicate_pairs(rows):
    """Exact matches stop; near matches need individual visual review and written reasons."""
    seen = {}
    for row in rows:
        if row['sha256'] in seen:
            first = seen[row['sha256']]
            raise ValueError('Exact duplicate: ' + first['subset'] + '/' + first['relativePath']
                             + ' and ' + row['subset'] + '/' + row['relativePath'])
        seen[row['sha256']] = row
    hashes = [(row, perceptual_hashes(row['path'])) for row in rows]
    flagged = []
    for i, (left, left_hashes) in enumerate(hashes):
        for right, right_hashes in hashes[i + 1:]:
            # Only cross-collection or cross-subset comparisons can leak into an evaluation.
            if left['subset'] == right['subset'] and left['subset'] in ('train', 'val', 'test'):
                continue
            if not (left['label'] == 'OOD' or right['label'] == 'OOD'):
                continue
            # Dihedral transformations of one image cover every relative rotation/mirror.
            distance = min((candidate ^ right_hashes[0]).bit_count() for candidate in left_hashes)
            if distance > 4:
                continue
            left_id = left['subset'] + '/' + left['relativePath'] + '/' + left['sha256']
            right_id = right['subset'] + '/' + right['relativePath'] + '/' + right['sha256']
            pair_id = hashlib.sha256('|'.join(sorted((left_id, right_id))).encode()).hexdigest()[:16]
            flagged.append({'pairId': pair_id, 'hammingDistance': distance,
                            'leftSubset': left['subset'], 'leftPath': left['relativePath'],
                            'leftSha256': left['sha256'], 'rightSubset': right['subset'],
                            'rightPath': right['relativePath'], 'rightSha256': right['sha256']})
    return flagged


def make_contact_sheets(rows, folder, prefix):
    """Every OOD photo gets an indexed thumbnail; mapping is saved as CSV."""
    folder.mkdir(parents=True, exist_ok=True)
    index = []
    for page_start in range(0, len(rows), 20):
        sheet = Image.new('RGB', (1000, 800), 'white')
        draw = ImageDraw.Draw(sheet)
        for offset, row in enumerate(rows[page_start:page_start + 20]):
            number = page_start + offset
            x, y = (offset % 5) * 200, (offset // 5) * 200
            with Image.open(row['path']) as picture:
                image = ImageOps.exif_transpose(picture).convert('RGB')
                image.thumbnail((190, 155))
                sheet.paste(image, (x + (200 - image.width) // 2, y))
            draw.text((x + 5, y + 158), f'{number:03d} {row["relativePath"][-24:]}', fill='black')
            index.append({'index': number, 'subset': row['subset'], 'relativePath': row['relativePath'],
                          'sha256': row['sha256']})
        sheet.save(folder / f'{prefix}_{page_start // 20:03d}.jpg', quality=88)
    write_csv_once(folder / f'{prefix}_index.csv', index)


def make_near_pair_sheets(pairs, rows, folder):
    """Place flagged images side by side so reviewers can resolve pair IDs visually."""
    locations = {(row['subset'], row['relativePath']): row['path'] for row in rows}
    for page_start in range(0, len(pairs), 5):
        sheet = Image.new('RGB', (1000, 1000), 'white')
        draw = ImageDraw.Draw(sheet)
        for offset, pair in enumerate(pairs[page_start:page_start + 5]):
            y = offset * 200
            for side, x in [('left', 0), ('right', 500)]:
                key = (pair[side + 'Subset'], pair[side + 'Path'])
                with Image.open(locations[key]) as picture:
                    image = ImageOps.exif_transpose(picture).convert('RGB')
                    image.thumbnail((485, 155))
                    sheet.paste(image, (x + (500 - image.width) // 2, y))
                draw.text((x + 5, y + 158), f"{pair['pairId']} {side}: {key[0]}/{key[1][-32:]}", fill='black')
        sheet.save(folder / f'near_pairs_{page_start // 5:03d}.jpg', quality=88)


def ood_identity(rows):
    return sorted((row['subset'], row['relativePath'], row['sha256']) for row in rows)


def verify_historical_ood(old_full, new_full, reference):
    # The original record pins content and fitting/report membership, not only folder names.
    if ood_identity(old_full) != ood_identity(reference['oldOod']):
        raise ValueError('Old OOD content or fitting/report membership changed from V4.1.1.')
    if ood_identity(new_full) != ood_identity(reference['newOod']):
        raise ValueError('The reused new OOD collection changed from the recorded V4.1.1 photos.')


## Step 7: Verify training and audit every photo

**Purpose:** Link this run to the completed combined model and the unchanged original split and OOD records.

**Reads:** training_complete.json, input_manifest.json, dataset_split.zip, arms/combined artifacts, historical OOD reference, both OOD folders.

**Saves:** A unique threshold run; temporary extracted mango split; audit/preflight.json; old/new contact sheets and near-pair sheets.

**Check:** Only the original seven-class per-class 70/15/15 split is accepted. Original OOD photos must match the pinned historical hashes; training backgrounds cannot overlap any OOD photo.

**Expected result:** Counts and audit folder print. Stop and inspect all contact sheets before Step 8.

In [ ]:
COMPLETE_PATH = TRAINING_RUN / 'training_complete.json'
if not COMPLETE_PATH.is_file():
    raise FileNotFoundError('Paste a completed V4.3.0 training run path in Step 1.')
complete = json.loads(COMPLETE_PATH.read_text(encoding='utf-8'))
COMPLETE_SHA256 = sha256_file(COMPLETE_PATH)
TRAIN_MANIFEST_PATH = TRAINING_RUN / 'input_manifest.json'
TRAIN_MANIFEST_SHA256 = sha256_file(TRAIN_MANIFEST_PATH)
if complete['inputManifestSha256'] != TRAIN_MANIFEST_SHA256 or complete['testPredictionsRun'] is not False:
    raise ValueError('Training manifest changed or mango test inference ran before gate fitting.')
if (set(complete['arms']) != set(ARMS) or complete['armOrder'] != list(ARMS)
        or complete['classes'] != CLASS_NAMES or complete['modelVersion'] != MODEL_VERSION):
    raise ValueError('Training must contain one combined seven-class V4.3.0 candidate.')
if sha256_file(TRAINING_RUN / 'dataset_split.zip') != complete['datasetSplitSha256']:
    raise ValueError('Dataset ZIP hash differs from the completed training run.')
if sha256_file(TRAINING_RUN / 'plan.json') != complete['planSha256']:
    raise ValueError('Training plan differs from the completed training record.')
verify_artifacts(TRAINING_RUN, complete['artifacts'])
manifest = json.loads(TRAIN_MANIFEST_PATH.read_text(encoding='utf-8'))
if manifest['classes'] != CLASS_NAMES or not manifest['images']:
    raise ValueError('Unexpected seven-class training split manifest.')
if set(r['subset'] for r in manifest['images']) != {'train', 'val', 'test'}:
    raise ValueError('The split must contain train, val, and test only.')
if any(r['label'] not in CLASS_NAMES for r in manifest['images']):
    raise ValueError('Unknown mango class in the split.')
if any(not PurePosixPath(r['relativePath']).as_posix().startswith(r['subset'] + '/')
       for r in manifest['images']):
    raise ValueError('Mango split path and subset disagree.')
split_counts = {label: {part: sum(r['label'] == label and r['subset'] == part
                                 for r in manifest['images'])
                        for part in ('train', 'val', 'test')}
                for label in CLASS_NAMES}
if any(min(counts.values()) == 0 for counts in split_counts.values()):
    raise ValueError('Every mango class needs train, validation, and test examples.')

arm_infos = {}
for arm in ARMS:
    arm_dir = TRAINING_RUN / 'arms' / arm
    info_path = arm_dir / 'resume_info.json'
    model_path = arm_dir / 'model_calibrated.keras'
    prob_path = arm_dir / 'validation_probabilities.npy'
    info = json.loads(info_path.read_text(encoding='utf-8'))
    if (info['arm'] != arm or info['modelVersion'] != MODEL_VERSION or info['classes'] != CLASS_NAMES
            or info['seed'] != SEED or info['testPredictionsRun'] is not False
            or info['inputManifestSha256'] != TRAIN_MANIFEST_SHA256
            or info['datasetSplitSha256'] != complete['datasetSplitSha256']
            or info['planSha256'] != complete['planSha256']
            or info['validationImageCount'] != sum(r['subset'] == 'val' for r in manifest['images'])
            or info['validationProbabilityOrder'] != 'input_manifest.json images subset=val in listed order'
            or info['modelSha256'] != sha256_file(model_path)
            or info['validationProbabilitiesSha256'] != sha256_file(prob_path)
            or info['temperatureFoldedIntoWeights'] is not True
            or info['preprocessing'] != PREPROCESSING
            or info['confidenceThreshold'] != CONFIDENCE_THRESHOLD
            or info['minimumValidationRetention'] != MIN_ID_RETENTION
            or not math.isfinite(float(info['temperature'])) or info['temperature'] <= 0
            or info['initialWeightsSha256'] != complete['initialWeightsSha256']):
        raise ValueError('Saved model or calibration provenance differs from the expected contract.')
    arm_complete = complete['arms'][arm]
    if {k: v for k, v in arm_complete.items() if k not in ('tfjsDirectory', 'tfjsArtifacts')} != info:
        raise ValueError('Completion metadata differs from resume_info.json.')
    for relative in ('resume_info.json', 'model_calibrated.keras', 'validation_probabilities.npy'):
        key = f'arms/{arm}/{relative}'
        if complete['artifacts'].get(key) != sha256_file(arm_dir / relative):
            raise ValueError('Training completion does not pin required artifact: ' + key)
    tfjs_files = verify_tfjs_export(arm_dir / 'tfjs_model')
    for relative in tfjs_files:
        key = f'arms/{arm}/tfjs_model/{relative}'
        if complete['artifacts'].get(key) != sha256_file(inside(arm_dir / 'tfjs_model', relative)):
            raise ValueError('Training completion does not pin browser file: ' + key)
    arm_infos[arm] = {**info, 'modelPath': str(model_path), 'infoPath': str(info_path),
                      'probPath': str(prob_path), 'infoSha256': sha256_file(info_path)}

if not REFERENCE_V411_INPUTS.is_file():
    raise FileNotFoundError('Restore the original V4.1.1 evaluation_inputs.json at the Step 1 path.')
if sha256_file(REFERENCE_V411_INPUTS) != REFERENCE_V411_SHA256:
    raise ValueError('Historical OOD reference changed. Restore the recorded original file.')
REFERENCE_V411_INPUTS_SHA256 = sha256_file(REFERENCE_V411_INPUTS)
reference_v411 = json.loads(REFERENCE_V411_INPUTS.read_text(encoding='utf-8'))
# This must be the original split, not a new random 70/15/15 partition.
def mango_identity(rows):
    return [(r['subset'], r['label'], r['relativePath'], r['sha256']) for r in rows]
if mango_identity(manifest['images']) != mango_identity(reference_v411['mango']):
    raise ValueError('Mango split membership/order/content differs from the original V4.1.1 seven-class split.')

# Audit exclusions after assigning the unchanged old OOD historical split.
old_full_rows, old_split_method = prepare_old_ood(OLD_OOD_DIR, [])
new_full_rows, new_all_paths = prepare_new_ood(NEW_OOD_DIR, [])
verify_historical_ood(old_full_rows, new_full_rows, reference_v411)
old_rows, checked_split_method = prepare_old_ood(OLD_OOD_DIR, OLD_OOD_EXCLUDE)
new_rows, _ = prepare_new_ood(NEW_OOD_DIR, NEW_OOD_EXCLUDE)
if checked_split_method != old_split_method:
    raise ValueError('Old OOD split changed while applying exclusions.')
expected_exclusions = {'old/' + p for p in OLD_OOD_EXCLUDE} | {'new/' + p for p in NEW_OOD_EXCLUDE}
if set(EXCLUSION_REASONS) != expected_exclusions:
    raise ValueError('EXCLUSION_REASONS must cover exactly the configured old/new exclusions.')
for key, reason in EXCLUSION_REASONS.items():
    if not isinstance(reason, str) or len(reason.strip()) < 12:
        raise ValueError('Record a content-based exclusion reason for ' + key)

run_id = 'v4-3-0-thresholds-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATASET_DIR = Path('/content/mangoleaf-threshold-v4-3-0') / run_id / 'dataset'
mango_records = extract_dataset(TRAINING_RUN / 'dataset_split.zip', manifest['images'], DATASET_DIR)
background_rows = []
for item in manifest.get('backgroundImages', []):
    source = Path(item['path'])
    if (not source.is_file() or sha256_file(source) != item['sha256']
            or source.stat().st_size != item['bytes']):
        raise RuntimeError('Training background source changed: ' + str(source))
    background_rows.append({**item, 'subset': 'background_source', 'label': 'BACKGROUND'})
# Audit all uploaded OOD images, so exclusions cannot hide overlap with training backgrounds.
all_records = mango_records + background_rows + old_full_rows + new_full_rows
verify_all_records_unchanged(all_records)
near_pairs = audit_duplicate_pairs(all_records)
AUDIT_DIR = RUN_DIR / 'audit'
AUDIT_DIR.mkdir(exist_ok=False)
make_contact_sheets(new_full_rows, AUDIT_DIR, 'new_ood')
make_contact_sheets(old_full_rows, AUDIT_DIR, 'old_ood')
make_near_pair_sheets(near_pairs, all_records, AUDIT_DIR)
if near_pairs:
    write_csv_once(AUDIT_DIR / 'near_duplicate_pairs.csv', near_pairs)
else:
    write_json_once(AUDIT_DIR / 'near_duplicate_pairs.json', [])
counts = {subset: sum(r['subset'] == subset for r in mango_records + background_rows + old_rows + new_rows)
          for subset in ['train', 'val', 'test', 'background_source', 'ood_tune', 'ood_report', 'ood_new_report']}
write_json_once(AUDIT_DIR / 'preflight.json', {
    'experimentVersion': EXPERIMENT_VERSION, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_SHA256, 'trainingManifestSha256': TRAIN_MANIFEST_SHA256,
    'referenceV411EvaluationInputsSha256': REFERENCE_V411_INPUTS_SHA256,
    'oldOodSplitMethod': old_split_method, 'oldOodExcluded': OLD_OOD_EXCLUDE,
    'newOodExcluded': NEW_OOD_EXCLUDE, 'exclusionReasons': EXCLUSION_REASONS,
    'counts': counts, 'perClassSplitCounts': split_counts,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_rows),
    'historicalNewOodCount': len(reference_v411['newOod']),
    'nearDuplicateFlagCount': len(near_pairs),
    'nearDuplicateThreshold': '64-bit DCT pHash, rotations/mirrors, Hamming distance <= 4',
    'runtime': RUNTIME, 'createdAt': datetime.now(timezone.utc).isoformat()})
print('Counts:', counts)
print('Per-class original split counts:', split_counts)
print('Reused new OOD uploaded:', len(new_all_paths), 'retained:', len(new_rows))
print('Possible near-duplicate pairs requiring review:', len(near_pairs))
print('Contact sheets and index:', AUDIT_DIR)
print('Output run:', RUN_DIR)


## Step 8: Review scope and freeze inputs

**Purpose:** Record human content and overlap decisions before any gate fitting.

**Reads:** Both complete OOD contact sheets/indexes and all flagged near-duplicate pair sheets.

**Saves:** evaluation_inputs.json and its SHA-256; retained validation, test, fitting, and report records in memory.

**Check:** Set both review flags only after reviewing every photo. Distinct near pairs need specific reasons. Actual OOD repeats that overlap retained training/fitting/report inputs must be resolved before starting a new audit run.

**Expected result:** Frozen input digest prints. The historically named new collection remains reused report-only evidence.

In [ ]:
OLD_OOD_SCOPE_REVIEWED = False
NEW_OOD_SCOPE_REVIEWED = False
SCOPE_REVIEW_NOTES = ''  # Describe both collections, mixed scenes, and every exclusion.
DISTINCT_NEAR_PAIR_REASONS = {}  # {'pairId': 'Visible reasons these are different photos/scenes'}
FRESH_COLLECTION_NOTES = 'Original V4.1/V4.1.1 OOD report photos reused; no pristine new OOD evidence in this run.'

if not OLD_OOD_SCOPE_REVIEWED or not NEW_OOD_SCOPE_REVIEWED or len(SCOPE_REVIEW_NOTES.strip()) < 20:
    raise RuntimeError('Review every old/new OOD contact-sheet entry and describe scope first.')
flag_ids = {pair['pairId'] for pair in near_pairs}
if set(DISTINCT_NEAR_PAIR_REASONS) != flag_ids or any(
        not isinstance(reason, str) or len(reason.strip()) < 12
        for reason in DISTINCT_NEAR_PAIR_REASONS.values()):
    raise RuntimeError('Review each possible near-duplicate pair; do not waive a true repeated photo.')
verify_all_records_unchanged(all_records)
INPUT_PATH = RUN_DIR / 'evaluation_inputs.json'
write_json_once(INPUT_PATH, {
    'experimentVersion': EXPERIMENT_VERSION, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_SHA256, 'trainingManifestSha256': TRAIN_MANIFEST_SHA256,
    'referenceV411EvaluationInputsSha256': REFERENCE_V411_INPUTS_SHA256,
    'modelVersion': MODEL_VERSION, 'armModelSha256': {a: arm_infos[a]['modelSha256'] for a in ARMS},
    'mango': manifest['images'], 'backgroundSources': background_rows,
    'oldOod': old_rows, 'newOod': new_rows,
    'oldOodAllUploaded': old_full_rows, 'newOodAllUploaded': new_full_rows,
    'oldOodExcluded': OLD_OOD_EXCLUDE, 'newOodExcluded': NEW_OOD_EXCLUDE,
    'oldOodSplitMethod': old_split_method,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_rows),
    'oldOodEvaluationStatus': 'exploratory reuse; historical fitting/report membership preserved',
    'newOodEvaluationStatus': 'V4.1/V4.1.1 reused photos; report-only after gate freeze, not pristine external test',
    'freshCollectionNotes': FRESH_COLLECTION_NOTES, 'scopeReviewNotes': SCOPE_REVIEW_NOTES,
    'distinctNearPairReasons': DISTINCT_NEAR_PAIR_REASONS,
    'exclusionReasons': EXCLUSION_REASONS, 'counts': counts,
    'reviewedAt': datetime.now(timezone.utc).isoformat()})
INPUT_SHA256 = sha256_file(INPUT_PATH)
val_records = [r for r in mango_records if r['subset'] == 'val']
test_records = [r for r in mango_records if r['subset'] == 'test']
tune_records = [r for r in old_rows if r['subset'] == 'ood_tune']
old_report_records = [r for r in old_rows if r['subset'] == 'ood_report']
new_report_records = new_rows
if min(map(len, [val_records, test_records, tune_records, old_report_records, new_report_records])) < 1:
    raise ValueError('Every fitting and report subset needs retained images.')
print('Input review frozen:', INPUT_PATH)
print('Input SHA-256:', INPUT_SHA256)


## Step 9: Fit and freeze the combined gate

**Purpose:** Choose margin/entropy rules with fixed calibrated weights using fitting data only.

**Reads:** Mango validation and old OOD fitting photos. Saved training validation probabilities verify inference parity.

**Saves:** selected_thresholds.json; selection_lock.json; arms/combined/threshold_search.csv and tuning_predictions.csv.

**Check:** There is no mango test, old OOD report, or reused new OOD inference here. The full search records feasible and infeasible grid points.

**Expected result:** One gate freezes with a durable SHA-256, or status is infeasible with null thresholds. A zero margin is valid.

In [ ]:
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
if SELECTION_PATH.exists() or (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('This run already fitted or evaluated. Preserve it; start a new run to change settings.')
verify_frozen_inputs()
selections = {}
search_rows = {}
tuning_rows = {}
for arm in ARMS:
    model = load_arm(arm)
    val_probabilities = predict_records(model, val_records)
    reference = np.load(arm_infos[arm]['probPath'], allow_pickle=False)
    if reference.shape != (len(val_records), len(CLASS_NAMES)):
        raise ValueError('Saved validation probability order/count differs: ' + arm)
    np.testing.assert_allclose(val_probabilities, reference, rtol=1e-4, atol=1e-4,
        err_msg='Saved validation predictions differ from this model or image loader: ' + arm)
    tune_probabilities = predict_records(model, tune_records)
    val_stats = probability_stats(val_probabilities.tolist())
    tune_stats = probability_stats(tune_probabilities.tolist())
    chosen, search = select_thresholds(val_stats, tune_stats, GRID,
                                        CONFIDENCE_THRESHOLD, MIN_ID_RETENTION)
    outcome = {'status': 'feasible' if chosen else 'infeasible', 'thresholds': chosen,
               'modelSha256': arm_infos[arm]['modelSha256'],
               'resumeInfoSha256': arm_infos[arm]['infoSha256'],
               'validationProbabilitiesSha256': arm_infos[arm]['validationProbabilitiesSha256'],
               'temperature': arm_infos[arm]['temperature'],
               'confidenceOnlyValidation': summarize(scored_rows(val_records, val_stats, CONFIDENCE_ONLY)),
               'confidenceOnlyOldOodTune': summarize(scored_rows(tune_records, tune_stats, CONFIDENCE_ONLY))}
    if chosen:
        outcome['validation'] = summarize(scored_rows(val_records, val_stats, chosen))
        outcome['oldOodTune'] = summarize(scored_rows(tune_records, tune_stats, chosen))
    selections[arm] = outcome
    search_rows[arm] = search
    tuning_rows[arm] = scored_rows(val_records, val_stats, chosen or CONFIDENCE_ONLY) + \
                        scored_rows(tune_records, tune_stats, chosen or CONFIDENCE_ONLY)
    print(arm, outcome['status'], chosen,
          'validation accepted:', outcome.get('validation', outcome['confidenceOnlyValidation'])['accepted'],
          'old OOD tune rejected:', outcome.get('oldOodTune', outcome['confidenceOnlyOldOodTune'])['rejected'])
    del model
    gc.collect()

verify_frozen_inputs()
selection = {'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION, 'classes': CLASS_NAMES, 'arms': selections,
    'armOrder': list(ARMS), 'confidenceOnlyReference': CONFIDENCE_ONLY,
    'minimumValidationRetention': MIN_ID_RETENTION, 'grid': GRID,
    'objective': 'maximum old OOD tune rejection with minimum mango validation retention',
    'tieBreak': 'maximum validation retention, minimum margin, maximum entropy',
    'trainingCompleteSha256': COMPLETE_SHA256, 'evaluationInputsSha256': INPUT_SHA256,
    'newOodUsedForFitting': False, 'testOrReportPredictionsBeforeSelection': False,
    'fittingSubsets': ['val', 'ood_tune'],
    'runtime': RUNTIME, 'preprocessing': PREPROCESSING,
    'selectedAt': datetime.now(timezone.utc).isoformat()}
write_json_once(SELECTION_PATH, selection)
SELECTION_SHA256 = sha256_file(SELECTION_PATH)
write_json_once(RUN_DIR / 'selection_lock.json', {
    'selectionSha256': SELECTION_SHA256, 'evaluationInputsSha256': INPUT_SHA256,
    'trainingCompleteSha256': COMPLETE_SHA256,
    'lockedBeforeReportInference': True, 'lockedAt': datetime.now(timezone.utc).isoformat()})
for arm in ARMS:
    arm_dir = RUN_DIR / 'arms' / arm
    arm_dir.mkdir(parents=True, exist_ok=False)
    write_csv_once(arm_dir / 'threshold_search.csv', search_rows[arm])
    write_csv_once(arm_dir / 'tuning_predictions.csv', tuning_rows[arm])
print('COMBINED GATE FROZEN:', SELECTION_PATH)
print('Selection SHA-256:', SELECTION_SHA256)


## Step 10: Evaluate the frozen gate

**Purpose:** Measure the fixed gate on report data after the selection lock exists.

**Reads:** Frozen selection and input hashes; mango test; old OOD report; reused new OOD report.

**Saves:** evaluation_started.json before inference, evaluation.json, evaluation_lock.json, raw_per_class_metrics.csv, full-precision prediction CSVs.

**Check:** Each photo is predicted once. The same saved scores feed selected and confidence-only reports. No thresholds change from these results.

**Expected result:** Raw seven-class accuracy, accepted correct/wrong, mango rejection, healthy/disease errors, and OOD false accepts are reported separately.

In [ ]:
if (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Evaluation already began. Keep this run intact; start another run if needed.')
frozen = read_selection_lock()
if (frozen['armOrder'] != list(ARMS) or frozen['evaluationInputsSha256'] != INPUT_SHA256 or
        frozen['trainingCompleteSha256'] != COMPLETE_SHA256 or
        frozen['newOodUsedForFitting'] is not False):
    raise RuntimeError('Frozen combined selection does not match the audited inputs.')
verify_frozen_inputs()
write_json_once(RUN_DIR / 'evaluation_started.json', {
    'selectionSha256': SELECTION_SHA256, 'inputSha256': INPUT_SHA256,
    'startedAt': datetime.now(timezone.utc).isoformat()})

from sklearn.metrics import classification_report, confusion_matrix
evaluations = {}
case_rows = {}
for arm in ARMS:
    selected = frozen['arms'][arm]
    if (selected['modelSha256'] != arm_infos[arm]['modelSha256'] or
            selected['resumeInfoSha256'] != arm_infos[arm]['infoSha256']):
        raise RuntimeError('Frozen gate belongs to different weights: ' + arm)
    model = load_arm(arm)
    test_probabilities = predict_records(model, test_records)
    old_report_probabilities = predict_records(model, old_report_records)
    new_report_probabilities = predict_records(model, new_report_records)
    stats_by_subset = {
        'mango_test': probability_stats(test_probabilities.tolist()),
        'old_ood_report': probability_stats(old_report_probabilities.tolist()),
        'new_ood_report': probability_stats(new_report_probabilities.tolist())}
    records_by_subset = {'mango_test': test_records, 'old_ood_report': old_report_records,
                         'new_ood_report': new_report_records}
    references = {'confidence_only': CONFIDENCE_ONLY}
    if selected['status'] == 'feasible':
        references['selected'] = selected['thresholds']
    arm_results = {}
    for gate_name, thresholds in references.items():
        reports = {}
        for subset, records_subset in records_by_subset.items():
            scored = scored_rows(records_subset, stats_by_subset[subset], thresholds)
            reports[subset] = summarize(scored)
            write_csv_once(RUN_DIR / 'arms' / arm / f'{gate_name}_{subset}_predictions.csv', scored)
            case_rows[(arm, gate_name, subset)] = scored
        arm_results[gate_name] = reports
    y_true = np.asarray([CLASS_NAMES.index(r['label']) for r in test_records])
    y_pred = test_probabilities.argmax(axis=1)
    raw_report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    raw_confusion = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    write_csv_once(RUN_DIR / 'arms' / arm / 'raw_per_class_metrics.csv',
                   [{'class': label, **raw_report[label]} for label in CLASS_NAMES])
    evaluations[arm] = {'status': selected['status'], 'thresholds': selected['thresholds'],
        'modelSha256': selected['modelSha256'], 'temperature': selected['temperature'],
        'rawClassificationReport': raw_report, 'rawConfusionMatrix': raw_confusion.tolist(),
        'calibratedTestCrossEntropy': float(-np.log(np.maximum(
            test_probabilities[np.arange(len(y_true)), y_true], 1e-7)).mean()),
        'gatedReports': arm_results}
    print(arm, 'raw test accuracy:', raw_report['accuracy'])
    for gate_name, reports in arm_results.items():
        print(' ', gate_name, 'wrong accepted:', reports['mango_test']['acceptedWrong'],
              'mango rejected:', reports['mango_test']['rejected'],
              'old OOD accepted:', reports['old_ood_report']['accepted'], '/', reports['old_ood_report']['total'],
              'new OOD accepted:', reports['new_ood_report']['accepted'], '/', reports['new_ood_report']['total'])
    del model
    gc.collect()

verify_frozen_inputs()
read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
evaluation = {'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'classes': CLASS_NAMES, 'armOrder': list(ARMS), 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_SHA256, 'arms': evaluations, 'selectionSha256': SELECTION_SHA256,
    'evaluationInputsSha256': INPUT_SHA256,
    'oldTestAndOodStatus': 'exploratory; used in earlier model development',
    'newOodStatus': 'V4.1/V4.1.1 reused photos; report-only after combined gate freeze, not pristine external test',
    'newOodCollectionNotes': FRESH_COLLECTION_NOTES,
    'newOodTotalUploaded': len(new_all_paths), 'newOodRetained': len(new_report_records),
    'oldOodTuneCount': len(tune_records), 'oldOodReportCount': len(old_report_records),
    'browserParityMeasured': False, 'freshLabeledMangoFieldAccuracyMeasured': False,
    'finishedAt': datetime.now(timezone.utc).isoformat()}
write_json_once(RUN_DIR / 'evaluation.json', evaluation)
EVALUATION_SHA256 = sha256_file(RUN_DIR / 'evaluation.json')
write_json_once(RUN_DIR / 'evaluation_lock.json', {
    'evaluationSha256': EVALUATION_SHA256, 'selectionSha256': SELECTION_SHA256,
    'evaluationInputsSha256': INPUT_SHA256, 'finishedAt': datetime.now(timezone.utc).isoformat()})
print('Evaluation saved:', RUN_DIR / 'evaluation.json')


## Step 11: Inspect one saved prediction

**Purpose:** Review a saved decision and all seven scores without running inference again.

**Reads:** A chosen Step 10 subset and saved selected or confidence-only decision.

**Saves:** Console preview only.

**Check:** Choose a valid image index. An infeasible run has only the confidence-only reference.

**Expected result:** Image path, original label, decision, rejection reasons, and class percentages print.

In [ ]:
DEBUG_ARM = 'combined'  # The only model trained by this workflow.
DEBUG_SUBSET = 'new_ood_report'  # mango_test, old_ood_report, new_ood_report
DEBUG_IMAGE_INDEX = 0
DEBUG_GATE = 'selected'  # selected or confidence_only

read_selection_lock()
if DEBUG_ARM not in ARMS or DEBUG_SUBSET not in records_by_subset:
    raise ValueError('Choose a saved arm and evaluated subset.')
if (DEBUG_ARM, DEBUG_GATE, DEBUG_SUBSET) not in case_rows:
    raise ValueError('This arm has no feasible selected gate; choose confidence_only.')
preview = case_rows[(DEBUG_ARM, DEBUG_GATE, DEBUG_SUBSET)]
if not isinstance(DEBUG_IMAGE_INDEX, int) or not 0 <= DEBUG_IMAGE_INDEX < len(preview):
    raise ValueError('DEBUG_IMAGE_INDEX is outside this saved subset.')
row = preview[DEBUG_IMAGE_INDEX]
print('Image:', row['relativePath'], '| actual:', row['label'])
print('Decision:', row['returnedClass'], '| rejection reasons:', row['rejectionReasons'] or 'none')
print_class_probabilities([row['probability_' + label] for label in CLASS_NAMES],
                          DEBUG_ARM + ' / ' + DEBUG_SUBSET)


## Step 12: Package the candidate and evidence

**Purpose:** Copy a feasible verified browser model unchanged and attach the fitted app-compatible configuration.

**Reads:** Frozen selection/evaluation, original training graph and manifest-listed shards, all run evidence.

**Saves:** Feasible candidate_models/combined/model_config.json and original browser model files; candidate hashes/ZIP; REPORT.md; threshold_complete.json; evidence ZIP.

**Check:** Copied graph/shard hashes must equal the pinned training hashes. An infeasible gate produces no candidate directory/ZIP. Temperature stays folded and is never applied twice.

**Expected result:** The full seven-class experimental candidate and evidence paths print. Return to the training notebook figure section with this threshold run path.

In [ ]:
frozen = read_selection_lock()
evaluation_lock = json.loads((RUN_DIR / 'evaluation_lock.json').read_text(encoding='utf-8'))
if (evaluation_lock['evaluationSha256'] != EVALUATION_SHA256
        or evaluation_lock['selectionSha256'] != SELECTION_SHA256
        or evaluation_lock['evaluationInputsSha256'] != INPUT_SHA256
        or sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_SHA256):
    raise RuntimeError('Frozen evaluation changed before packaging.')
verify_frozen_inputs()
candidate_root = RUN_DIR / 'candidate_models'
candidate_paths = {}
candidate_hashes = {}
for arm in ARMS:
    selected = frozen['arms'][arm]
    if selected['status'] != 'feasible':
        if selected['thresholds'] is not None:
            raise ValueError('An infeasible gate cannot have candidate thresholds.')
        print('No fitted candidate exported:', arm, 'is infeasible.')
        continue
    source = TRAINING_RUN / 'arms' / arm / 'tfjs_model'
    model_files = verify_tfjs_export(source)
    source_hashes = {}
    for relative in model_files:
        trained_relative = f'arms/{arm}/tfjs_model/{relative}'
        source_hashes[relative] = sha256_file(inside(source, relative))
        if complete['artifacts'].get(trained_relative) != source_hashes[relative]:
            raise RuntimeError('Browser export changed after training: ' + trained_relative)
    candidate_dir = candidate_root / arm
    candidate_dir.mkdir(parents=True, exist_ok=False)
    for relative in model_files:
        destination = inside(candidate_dir, relative)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(inside(source, relative), destination)
        if sha256_file(destination) != source_hashes[relative]:
            raise RuntimeError('Copied browser file hash differs from original: ' + relative)
    config = {'modelVersion': MODEL_VERSION, 'experimentVersion': EXPERIMENT_VERSION, 'arm': arm,
        'classes': CLASS_NAMES, 'inputSize': list(IMAGE_SIZE), 'preprocessing': PREPROCESSING,
        'temperature': selected['temperature'], 'temperatureFoldedIntoWeights': True,
        'thresholds': selected['thresholds'], 'status': 'experimental candidate; not adopted or deployed',
        'modelSha256': selected['modelSha256'], 'selectionSha256': SELECTION_SHA256,
        'evaluationSha256': EVALUATION_SHA256, 'trainingCompleteSha256': COMPLETE_SHA256,
        'browserParityMeasured': False}
    if (config['classes'] != CLASS_NAMES or len(config['classes']) != 7
            or config['thresholds']['confidenceThreshold'] != 0.70
            or config['temperatureFoldedIntoWeights'] is not True):
        raise ValueError('Candidate configuration differs from the app seven-class contract.')
    write_json_once(candidate_dir / 'model_config.json', config)
    copied_files = verify_tfjs_export(candidate_dir)
    if copied_files != model_files:
        raise RuntimeError('Candidate weight manifest differs after copying.')
    verify_artifacts(candidate_dir, source_hashes)
    candidate_hashes[arm] = {relative: sha256_file(inside(candidate_dir, relative))
                            for relative in model_files + ['model_config.json']}
    candidate_zip = RUN_DIR / f'{RUN_DIR.name}-{arm}-candidate_model.zip'
    with zipfile.ZipFile(candidate_zip, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
        for relative in model_files + ['model_config.json']:
            bundle.write(inside(candidate_dir, relative), relative)
    # Read the ZIP back: the downloadable candidate must equal the verified copied files.
    with zipfile.ZipFile(candidate_zip) as bundle:
        if set(bundle.namelist()) != set(candidate_hashes[arm]):
            raise RuntimeError('Candidate ZIP entries differ from the verified file list.')
        for relative, expected in candidate_hashes[arm].items():
            if hashlib.sha256(bundle.read(relative)).hexdigest() != expected:
                raise RuntimeError('Candidate ZIP changed file content: ' + relative)
    candidate_paths[arm] = str(candidate_zip)
write_json_once(RUN_DIR / 'candidate_hashes.json', candidate_hashes)
write_json_once(RUN_DIR / 'candidate_exports.json', {
    'status': frozen['arms']['combined']['status'], 'candidateZips': candidate_paths,
    'candidateZipSha256': {arm: sha256_file(path) for arm, path in candidate_paths.items()},
    'trainingFilesCopiedUnchanged': bool(candidate_paths),
    'selectionSha256': SELECTION_SHA256, 'evaluationSha256': EVALUATION_SHA256})

report_lines = [
    '# MangoLeaf AI V4.3.0: combined seven-class threshold experiment', '',
    'Experimental candidate evidence. No adoption, deployment, browser parity, or field accuracy is claimed.',
    f'Training run: `{TRAINING_RUN}`', f'Threshold run: `{RUN_DIR}`',
    f'Training completion SHA-256: `{COMPLETE_SHA256}`',
    f'Selection SHA-256: `{SELECTION_SHA256}`', f'Evaluation SHA-256: `{EVALUATION_SHA256}`', '',
    'Gate fitting keeps calibrated neural-network weights fixed; it is not model training.',
    'Confidence stayed at 0.70. Margin/entropy were fitted only on validation + historical old OOD fitting photos, requiring at least 95% validation retention.',
    'The gate and its hash were frozen before mango test or OOD report inference.',
    'Mango test and old OOD report reuse is exploratory. The historically named new OOD folder is reused V4.1/V4.1.1 report evidence, not a pristine external test.',
    'Training background sources and both OOD collections were audited separately; OOD photos are not training augmentation inputs.', '',
    f'Counts: validation {len(val_records)}, mango test {len(test_records)}, old OOD fitting {len(tune_records)}, old OOD report {len(old_report_records)}, reused new OOD uploaded {len(new_all_paths)}, retained {len(new_report_records)}.',
    f'Scope review: {SCOPE_REVIEW_NOTES}', f'Collection history: {FRESH_COLLECTION_NOTES}',
    f'Audited exclusions: {json.dumps(EXCLUSION_REASONS, sort_keys=True)}', '',
    '| Model | Gate | Test correct accepted | Test wrong accepted | Mango rejected | Old OOD accepted | Reused new OOD accepted |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: |']
for arm in ARMS:
    result = evaluations[arm]
    for gate_name, reports in result['gatedReports'].items():
        mango, old_ood, new_ood = (reports[k] for k in ('mango_test', 'old_ood_report', 'new_ood_report'))
        report_lines.append(f'| {arm} | {gate_name} | {mango["acceptedCorrect"]}/{mango["total"]} | '
            f'{mango["acceptedWrong"]} | {mango["rejected"]} | '
            f'{old_ood["accepted"]}/{old_ood["total"]} | {new_ood["accepted"]}/{new_ood["total"]} |')
report_lines += ['', f'Fitted gate status: **{frozen["arms"]["combined"]["status"]}**.',
    'An infeasible gate is not exported as a valid candidate. Confidence-only figures remain reference evidence.',
    'Validation retention counts accepted photos, including accepted wrong diagnoses; it does not measure accuracy.',
    'Full precision probabilities, rejection reasons, seven-class metrics, raw/gated confusion, and healthy/disease/organ errors are saved in evaluation.json and arms/combined CSVs.',
    'Continue in the matching training notebook for curves, confusion matrices, per-class metrics, threshold search, and Grad-CAM from this same frozen run.',
    'Any decisions informed by reused OOD reports need another untouched OOD collection for final independent evaluation. Expert-labeled field mango photos and real browser checks are still needed.']
with (RUN_DIR / 'REPORT.md').open('x', encoding='utf-8') as handle:
    handle.write('\n'.join(report_lines) + '\n')

# Export evidence hashes before archiving; keep full training and threshold Drive folders.
evidence_files = [p for p in sorted(RUN_DIR.rglob('*'))
                  if p.is_file() and p.suffix != '.zip' and not p.is_relative_to(candidate_root)]
write_json_once(RUN_DIR / 'evidence_hashes.json', {
    p.relative_to(RUN_DIR).as_posix(): sha256_file(p) for p in evidence_files})
completion_files = evidence_files + [RUN_DIR / 'evidence_hashes.json']
write_json_once(RUN_DIR / 'threshold_complete.json', {
    'experimentVersion': EXPERIMENT_VERSION, 'modelVersion': MODEL_VERSION,
    'classes': CLASS_NAMES, 'armOrder': list(ARMS), 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_SHA256, 'selectedThresholdsSha256': SELECTION_SHA256,
    'evaluationSha256': EVALUATION_SHA256, 'evaluationInputsSha256': INPUT_SHA256,
    'referenceV411EvaluationInputsSha256': REFERENCE_V411_INPUTS_SHA256,
    'fittedGateStatus': frozen['arms']['combined']['status'],
    'status': 'completed threshold evidence; experimental candidate, not adopted or deployed',
    'artifacts': {p.relative_to(RUN_DIR).as_posix(): sha256_file(p) for p in completion_files},
    'finishedAt': datetime.now(timezone.utc).isoformat()})
evidence_zip = RUN_DIR / f'{RUN_DIR.name}-evidence.zip'
with zipfile.ZipFile(evidence_zip, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in completion_files + [RUN_DIR / 'threshold_complete.json']:
        bundle.write(path, path.relative_to(RUN_DIR).as_posix())
    for name in ('training_complete.json', 'input_manifest.json', 'plan.json'):
        bundle.write(TRAINING_RUN / name, 'training/' + name)
    for arm in ARMS:
        for name in ('resume_info.json', 'training_history.json'):
            path = TRAINING_RUN / 'arms' / arm / name
            if path.is_file():
                bundle.write(path, f'training/arms/{arm}/{name}')
    bundle.write(REFERENCE_V411_INPUTS, 'historical/v4.1.1_evaluation_inputs.json')
verify_frozen_inputs()
read_selection_lock()
print('\n'.join(report_lines))
print('THRESHOLD_RUN for the training notebook figure section:', RUN_DIR)
print('Threshold completion SHA-256:', sha256_file(RUN_DIR / 'threshold_complete.json'))
print('Evidence ZIP:', evidence_zip, '| SHA-256:', sha256_file(evidence_zip))
for arm, path in candidate_paths.items():
    print('Experimental candidate ZIP:', arm, path, '| SHA-256:', sha256_file(path))


## Saved files and next action

`selected_thresholds.json`, `selection_lock.json`, and `evaluation_inputs.json` identify the fixed gate and inputs. `evaluation.json` and `evaluation_lock.json` identify the completed reports. `threshold_complete.json` links training, selection, evaluation, and hashes of every figure-input evidence artifact. `arms/combined/` contains threshold-search, tuning, raw per-class metrics, and full-precision `confidence_only_*` and feasible `selected_*` prediction CSVs. `audit/` contains scope and overlap evidence, including every uploaded OOD photo and the documented exclusions.

A feasible run has one `candidate_models/combined/` directory with unchanged `model.json` and manifest-listed shards plus `model_config.json`. `candidate_hashes.json` pins each copied file; `candidate_exports.json` records ZIP hashes. Infeasible runs save empty candidate hashes, no candidate directory, and no candidate model ZIP. `REPORT.md`, `evidence_hashes.json`, and the evidence ZIP preserve the fitting and reporting record.

Paste the printed **THRESHOLD_RUN** into `MangoLeaf_AI_Training_v4_3_0.ipynb` and run its remaining figure steps. That notebook owns the combined training curves, augmentation preview, confusion, per-class metrics, threshold search, and Grad-CAM. Keep all figures linked to these run hashes. The evidence ZIP omits large model weights and the mango dataset ZIP; preserve the complete original Drive folders.

This template establishes a procedure, not measured model quality. Do not copy candidate files to the application or update shipped version labels from this notebook.